In [ ]:
# Diagnóstico del Error de Build TSX

Este notebook analiza un error de compilación de Next.js en `src/components/screens/ScreenAjustes.tsx`, localizado en el cierre de un componente modal y antes de una declaración de tipo TS.
</VSCode.Cell>
<VSCode.Cell language="python">
from pathlib import Path
project_root = Path('D:/Proyecto Web/APP Barberia')
file_path = project_root / 'src/components/screens/ScreenAjustes.tsx'
print('Proyecto:', project_root)
print('Archivo a inspeccionar:', file_path)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Reproducir el Error

Mensaje de error reportado:

```
× Unexpected token. Did you mean `{'}'}` or `&rbrace;`?

./src/components/screens/ScreenAjustes.tsx
Error:   × Unexpected token. Did you mean `{'}'}` or `&rbrace;`?
     ╭─[D:\Proyecto Web\APP Barberia\src\components\screens\ScreenAjustes.tsx:792:1]
 789 │       </div>
 790 │     </div>
 791 │   );
 792 │ }
     · ▲
 793 │ 
 794 │ // ─── Modal Editar Socio ─────────────────────────────────────────────────────────
 795 │ type ModalEditarSocioProps = { socio: Socio; onClose: () => void };
     ╰────
  × Expected '</', got ':'
```

Esto sugiere que el parser sigue dentro de un bloque JSX cuando llega a la declaración de tipo TS.
</VSCode.Cell>
<VSCode.Cell language="python">
text = file_path.read_text(encoding='utf-8')
lines = text.splitlines()
for i in range(780, 806):
    print(f'{i+1:4}: {lines[i]}')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Inspección del Bloque TSX

Vamos a revisar el código alrededor del final de `ModalGestionSocios` y el comienzo de `ModalEditarSocio` para encontrar cierres faltantes o expresiones JSX abiertas.
</VSCode.Cell>
<VSCode.Cell language="python">
region = '\n'.join(lines[760:806])
print(region)
print('\n--- contadores de delimitadores ---')
for ch in ['{','}','(',')','<','>']:
    print(f'{ch}: {region.count(ch)}')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Observaciones

Si el parser mantiene el modo JSX al llegar a la declaración `type ModalEditarSocioProps`, normalmente significa una etiqueta JSX abierta sin cerrar dentro del componente anterior.

Buscaremos etiquetas con apertura `<` que no tengan cierre, expresiones ternarias mal formadas en atributos, o un `</div>` faltante en la estructura del retorno.
</VSCode.Cell>
<VSCode.Cell language="python">
from collections import Counter
open_tags = Counter()
for line in region.splitlines():
    if '<div' in line: open_tags['div_open'] += 1
    if '</div>' in line: open_tags['div_close'] += 1
    if '<button' in line: open_tags['button_open'] += 1
    if '</button>' in line: open_tags['button_close'] += 1
    if '<span' in line: open_tags['span_open'] += 1
    if '</span>' in line: open_tags['span_close'] += 1
print(open_tags)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Corrección propuesta

La corrección puede implicar cualquiera de los siguientes cambios:

1. Cerrar una etiqueta JSX abierta dentro del `return` de `ModalGestionSocios`.
2. Reemplazar la destructuración tipada de parámetros con una declaración de props separada si hay problemas de parseo en TSX.
3. Asegurarse de que todos los operadores ternarios dentro de `style` u otros atributos estén envueltos correctamente.
</VSCode.Cell>
<VSCode.Cell language="python">
for i in range(792, 800):
    print(f'{i+1:4}: {lines[i]}')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Ejemplo de componente correcto

Un componente TSX válido con type alias debe verse así:

```tsx
type ExampleProps = { value: string };
function Example({ value }: ExampleProps) {
  return (
    <div>{value}</div>
  );
}
```

La clave es que la declaración de tipo esté fuera del contexto JSX y que cualquier bloque JSX quede cerrado antes de terminar la función.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Validación final

Si el análisis revela un cierre faltante, el siguiente paso será aplicar la corrección directamente en `ScreenAjustes.tsx` y volver a ejecutar `npm run build`.
